# Getting started: how to get data from MANTRA

This notebook is a hands-on tour of the different ways to get data out of
the [MANTRA](https://github.com/aidos-lab/mantra) dataset. MANTRA is a
collection of *combinatorial triangulations* of 2- and 3-manifolds,
together with precomputed topological invariants.

We cover:

1. Loading the dataset and inspecting a single triangulation
2. The raw fields available on every sample
3. Iterating over and filtering the dataset with plain Python
4. Turning triangulations into **graphs** (the 1-skeleton) and other
   combinatorial **representations**
5. Attaching **features** (random / degree / geometric embeddings)
6. Creating **labels** for classification tasks
7. Assembling a **model-ready** dataset and batching it with a PyG
   `DataLoader`
8. Building **out-of-distribution** test sets with barycentric, stellar,
   and graded **subdivisions** (`MANTRADivided`)
9. Loading other variants: 3-manifolds, the balanced dataset, and
   pinned versions

Everything here uses only the public `mantra` API.

## 0. Installation

The dataset loader is published on PyPI. Uncomment the line below if you
are running this notebook outside of a checkout of the repository.

`pip install mantra-dataset` also pulls in
[`pytorch-geometric`](https://pytorch-geometric.readthedocs.io/), whose
`Data`/`InMemoryDataset` API MANTRA follows.


If you are working on a different branch than main, you should use clone the branch and work in it

In [40]:
# %pip install mantra-dataset

## 1. Load the dataset

`ManifoldTriangulations` follows the `torch_geometric` `InMemoryDataset`
convention: point it at a `root` folder and it will download the release,
cache a processed copy, and load it into memory. The first call downloads
and processes the data; subsequent calls reuse the cache.

In [41]:
from mantra.datasets import ManifoldTriangulations

dataset = ManifoldTriangulations(
    root="./data",      # where the raw + processed data is stored
    dimension=2,        # 2-manifolds (surfaces); use 3 for 3-manifolds
    version="latest",   # a released version, or "latest"
    force_reload=True
)

print(f"Loaded {len(dataset)} triangulations of 2-manifolds.")

Extracting data/mantra/v0.0.19/2D/raw/2_manifolds.json.gz
Processing...
Done!


Loaded 43214 triangulations of 2-manifolds.


## 2. Inspect a single triangulation

Indexing the dataset returns a `torch_geometric.data.Data` object. Its
`repr` lists the available attributes and their shapes.

In [42]:
sample = dataset[0]
sample

Data(id='manifold_2_4_5_1', triangulation=[4], dimension=[1], n_vertices=[1], betti_numbers=[3], torsion_coefficients=[3], name='S^2', genus=[1], orientable=[1], vertex_transitive=[1])

## 3. The raw fields

Every sample carries the raw triangulation plus a set of precomputed
topological invariants. The most important fields are:

| field | meaning |
| --- | --- |
| `id` | unique identifier of the triangulation |
| `name` | homeomorphism type, e.g. `S^2`, `T^2`, `Klein bottle` (empty string if unnamed) |
| `dimension` | intrinsic dimension of the manifold (2 or 3) |
| `n_vertices` | number of vertices |
| `triangulation` | list of top-dimensional simplices, using **1-indexed** vertices |
| `betti_numbers` | Betti numbers $[\beta_0, \beta_1, \dots]$ |
| `torsion_coefficients` | torsion of the homology groups |
| `orientable` | whether the manifold is orientable |
| `genus` | genus of the surface (for 2-manifolds) |

In [43]:
print("id:                  ", sample.id)
print("name:                ", sample.name)
print("dimension:           ", int(sample.dimension))
print("n_vertices:          ", int(sample.n_vertices))
print("triangulation:       ", sample.triangulation)
print("betti_numbers:       ", sample.betti_numbers)
print("torsion_coefficients:", sample.torsion_coefficients)
print("orientable:          ", bool(sample.orientable))
print("genus:               ", int(sample.genus))

id:                   manifold_2_4_5_1
name:                 S^2
dimension:            2
n_vertices:           4
triangulation:        [[1, 2, 3], [1, 2, 4], [1, 3, 4], [2, 3, 4]]
betti_numbers:        [1, 0, 1]
torsion_coefficients: ['', '', '']
orientable:           True
genus:                0


The `triangulation` is just a nested Python list of the top-dimensional
simplices. For a 2-manifold these are triangles; the four triangles above
are the boundary of a tetrahedron, i.e. the smallest triangulation of the
2-sphere $S^2$. Vertices are numbered starting from 1.

## 4. Iterate and filter with plain Python

Because the dataset is a plain sequence of `Data` objects, you can treat
it like any Python iterable — no transforms required. For example, we can
count how many triangulations exist per homeomorphism type, or pull out
all triangulations of the torus.

In [44]:
from collections import Counter

names = Counter(d.name if d.name else "(unnamed)" for d in dataset)
print("Most common homeomorphism types:")
for name, count in names.most_common(11):
    print(f"  {name:14s} {count}")

tori = [d for d in dataset if d.name == "T^2"]
print(f"\nThere are {len(tori)} triangulations of the torus T^2.")

Most common homeomorphism types:
  #^4 RP^2       13695
  #^3 RP^2       11917
  #^5 RP^2       7053
  Klein bottle   4657
  T^2            2247
  RP^2           1365
  #^6 RP^2       1022
  #^2 T^2        868
  S^2            308
  #^3 T^2        23
  #^7 RP^2       16

There are 2247 triangulations of the torus T^2.


## 5. Turn a triangulation into a graph (the 1-skeleton)

Most graph models want an `edge_index`. The `OneSkeleton` representation
builds the **1-skeleton** of the triangulation (vertices + edges) and
exposes it as a 0-indexed `edge_index`. Pass it as the dataset's
`transform` so it is applied lazily on access.

In [45]:
from mantra.representations import OneSkeleton

graph_dataset = ManifoldTriangulations(
    root="./data",
    dimension=2,
    version="latest",
    transform=OneSkeleton(), # also try DualGraph(), HasseDiagram()
)

g = graph_dataset[0]
print("num_nodes:       ", int(g.num_nodes))
print("edge_index shape:", tuple(g.edge_index.shape))
print(g.edge_index)

num_nodes:        4
edge_index shape: (2, 12)
tensor([[0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3],
        [1, 2, 3, 0, 2, 3, 0, 1, 3, 0, 1, 2]])


## 6. Attach features

MANTRA triangulations are purely combinatorial — vertices have no
intrinsic coordinates. The feature transforms in `mantra.transforms`
give you input features to work with. Transforms compose with
`torch_geometric.transforms.Compose`.

- `NodeRandomTransform(dim=k)` → random `random_features` of shape
  `(num_nodes, k)`
- `NodeDegreeTransform()` → per-vertex `degree` of shape `(num_nodes, 1)`

(Both require the graph, so we run `OneSkeleton` first.)

In [46]:
from torch_geometric.transforms import Compose
from mantra.transforms import NodeRandomTransform, NodeDegreeTransform

feat_dataset = ManifoldTriangulations(
    root="./data",
    dimension=2,
    version="latest",
    transform=Compose(
        [
            OneSkeleton(),
            NodeRandomTransform(dim=8),
            NodeDegreeTransform(),
        ]
    ),
)

f = feat_dataset[0]
print("random_features:", tuple(f.random_features.shape))
print("degree:         ", tuple(f.degree.shape))
print("degrees:        ", f.degree.view(-1))

random_features: (4, 8)
degree:          (4, 1)
degrees:         tensor([3., 3., 3., 3.])


## 7. Create labels for a task

`CreateLabels(source=...)` derives an integer classification target `y`
from any attribute. For a nominal attribute such as `name`, each distinct
value is mapped to an integer class index.

In [47]:
from mantra.transforms import CreateLabels

by_name = ManifoldTriangulations(
    root="./data",
    dimension=2,
    version="latest",
    transform=CreateLabels(source="name"),
)

for i in [0, 5, 10]:
    d = by_name[i]
    print(f"{d.name:14s} -> y = {int(d.y)}")

S^2            -> y = 0
T^2            -> y = 1
T^2            -> y = 1


`orientable` is a binary property. You can either read the raw boolean
directly for a binary task, or let `CreateLabels` turn it into a `{0, 1}`
target (the integer index is assigned in order of appearance, so it is not
guaranteed that `True` maps to `1`).

In [48]:
orient = ManifoldTriangulations(
    root="./data",
    dimension=2,
    version="latest",
    transform=CreateLabels(source="orientable"),
)
print("raw orientable flag:", bool(orient[0].orientable))
print("integer label y:    ", int(orient[0].y))

raw orientable flag: True
integer label y:     0


## 8. Other representations

Beyond the 1-skeleton, MANTRA can express a triangulation through several
other combinatorial objects (all in `mantra.representations`):

- `DualGraph` — the dual graph (top simplices as nodes)
- `HasseDiagram` — the face lattice of the complex
- `AdjacencySimplicialComplex` / `CoadjacencySimplicialComplex` /
  `IncidenceSimplicialComplex` — the (co)adjacency and incidence matrices
  of each dimension, for higher-order / simplicial networks

The simplicial-complex transforms take a `signed` flag and store one
tensor per dimension (`adjacency_0`, `adjacency_1`, ...).

In [51]:
from mantra.representations import (
    DualGraph,
    HasseDiagram,
    AdjacencySimplicialComplex,
    IncidenceSimplicialComplex,
)

dual = ManifoldTriangulations(
    root="./data", dimension=2, version="latest", transform=DualGraph()
)[0]
print("DualGraph edge_index:   ", tuple(dual.edge_index.shape))

hasse = ManifoldTriangulations(
    root="./data", dimension=2, version="latest", transform=HasseDiagram()
)[0]
print("HasseDiagram edge_index:", tuple(hasse.edge_index.shape))

adj = ManifoldTriangulations(
    root="./data",
    dimension=2,
    version="latest",
    transform=AdjacencySimplicialComplex(signed=False),
)[0]
print("adjacency_0 (vertices): ", tuple(adj.adjacency_0.shape))
print("adjacency_1 (edges):    ", tuple(adj.adjacency_1.shape))

inc = ManifoldTriangulations(
    root="./data",
    dimension=2,
    version="latest",
    transform=IncidenceSimplicialComplex(signed=True),
)[0]
print("incidence_1 (vtx->edge):", tuple(inc.incidence_1.shape))

DualGraph edge_index:    (2, 12)
HasseDiagram edge_index: (2, 48)
adjacency_0 (vertices):  (4, 4)
adjacency_1 (edges):     (6, 6)
incidence_1 (vtx->edge): (4, 6)


### Geometric embeddings

If you need actual coordinates, `MomentCurveEmbedding` places vertices on
the moment curve — a canonical embedding derived purely from the number
of vertices and the dimension.

In [49]:
from mantra.transforms import MomentCurveEmbedding

mc = ManifoldTriangulations(
    root="./data",
    dimension=2,
    version="latest",
    transform=MomentCurveEmbedding(),
)[0]
print("moment_curve_embedding:", tuple(mc.moment_curve_embedding.shape))

moment_curve_embedding: (4, 5)


## 9. A model-ready dataset + batching

Putting it together: build the graph, add features, create a label, and
finally use `SelectAttributes` to keep only the tensors a model needs
(`x`, `edge_index`, `y`). The result batches cleanly with a PyG
`DataLoader`.

In [53]:
from mantra.transforms import SelectFeatures, SelectAttributes
from torch_geometric.loader import DataLoader

model_ready = ManifoldTriangulations(
    root="./data",
    dimension=2,
    version="latest",
    transform=Compose(
        [
            OneSkeleton(),
            NodeRandomTransform(dim=8),
            # store random features under the canonical `x` attribute
            SelectFeatures(src="random_features", dst="x"),
            CreateLabels(source="orientable"),
            # drop everything except the tensors below
            SelectAttributes(keep_keys=["x", "edge_index", "y"]),
        ]
    ),
)

print("model-ready sample:", model_ready[0])

loader = DataLoader(model_ready, batch_size=32, shuffle=True)
batch = next(iter(loader))
print("batch:            ", batch)
print("x:                ", tuple(batch.x.shape))
print("y:                ", tuple(batch.y.shape))
print("graphs in batch:  ", batch.num_graphs)

model-ready sample: Data(edge_index=[2, 12], x=[4, 8], y=[1])
batch:             DataBatch(edge_index=[2, 2184], x=[320, 8], y=[32], batch=[320], ptr=[33])
x:                 (320, 8)
y:                 (32,)
graphs in batch:   32


## 10. Out-of-distribution test sets via subdivision

A central question in topological deep learning is whether a model has
learned the *topology* of a triangulation or merely memorised triangulations
of a certain size. `MANTRADivided` builds a benchmark for exactly this: it
splits the data into `train` / `val` / `test`, and then produces an extra
**`ood`** (out-of-distribution) split by **subdividing the test
triangulations**.

Subdivision refines a triangulation into more (smaller) simplices **without
changing its homeomorphism type** — so every label (`name`, `betti_numbers`,
`orientable`, ...) is identical to the corresponding test sample, but the
triangulation is finer and larger than anything seen during training. A model
that has genuinely learned topology should do as well on `ood` as on `test`.

Three subdivision schemes are available through `division_type`:

| `division_type` | what it does | key argument |
| --- | --- | --- |
| `"barycentric"` | full barycentric subdivision (the order complex) | `round` — how many times to repeat |
| `"stellar"` | 1-(d+1) Pachner move on a fraction of the top simplices | `fraction` in `[0, 1]` |
| `"graded"` | random stellar moves until a vertex target is reached | `min_vertices` — target vertex count |

Each of the four splits is retrieved by passing the matching `split_type`.

In [50]:
from mantra.datasets import MANTRADivided

# Build the four splits for a single(default) barycentric OOD test set. They share the same
# processing, so only the first call does the (one-off) heavy lifting of
# splitting and subdividing.
common = dict(
    root="./data", dimension=2, version="latest", division_type="barycentric", round=1
)

train = MANTRADivided(split_type="train", **common)
val = MANTRADivided(split_type="val", **common)
test = MANTRADivided(split_type="test", **common)
ood = MANTRADivided(split_type="ood", **common)

print(f"train: {len(train):6d}")
print(f"val:   {len(val):6d}")
print(f"test:  {len(test):6d}")
print(f"ood:   {len(ood):6d}   (barycentric subdivisions of the test split)")

Processing...
Subdividing OOD: 100%|██████████| 8643/8643 [00:02<00:00, 3967.14it/s]
Done!


train:  25928
val:     8643
test:    8643
ood:     8643   (barycentric subdivisions of the test split)


In [51]:
# The i-th OOD sample is the subdivision of the i-th test sample: the same
# manifold (identical label), but a finer, larger triangulation.
t, o = test[0], ood[0]
print(f"test[0]:  name={t.name!r:12s}  n_vertices={int(t.n_vertices)}")
print(f"ood[0]:   name={o.name!r:12s}  n_vertices={int(o.n_vertices)}")
print("same homeomorphism type:", t.name == o.name)

test[0]:  name='#^3 RP^2'    n_vertices=10
ood[0]:   name='#^3 RP^2'    n_vertices=65
same homeomorphism type: True


### Balancing and filtering the splits

As seen in section 4, MANTRA's class distribution is heavily skewed (13695
copies of `#^4 RP^2` vs. 23 of `#^3 T^2`). `MANTRADivided` accepts several
keywords to counter this and to control what ends up in each split:

| keyword | effect |
| --- | --- |
| `balanced=True` | balance the *whole* dataset before splitting, using Pachner-move augmentation + deduplication |
| `balance_kwargs=dict(...)` | tune the balancing; allowed keys: `target_count`, `n_moves`, `use_topology_changes`, `verbose` (the seed and vertex cap come from the top-level `seed` and `max_vertices`) |
| `max_vertices=n` | keep only triangulations with at most `n` vertices in train/val/test; with `balanced=True` the cap is enforced inside the balancing itself, so the classes stay balanced under the cap. Combined with a graded subdivision (`vertex_number > n`) every OOD sample is strictly larger than anything seen in-distribution |
| `class_count_filter=n` | drop all classes with fewer than `n` triangulations before splitting |
| `max_ood_size_per_class=n` | oversample or trim the OOD split so that every class contains exactly `n` samples |
| `stratified=True` | stratify the train/val/test split by class name |
| `split_proportions=[a, b, c]` | change the train/val/test proportions (default `[0.6, 0.2, 0.2]`) |

Three caveats:

- Each Pachner move may add a vertex, so under a vertex cap a class is
  only balanceable if it has sources with at most
  `max_vertices - n_moves` vertices — otherwise the balancing raises a
  `ValueError` telling you to lower `n_moves` or raise the cap.
- `balanced=True` equalizes the classes **before** `class_count_filter` is
  applied, so that filter can re-imbalance the data or drop classes again
  (a `UserWarning` reminds you of this). It is rarely useful together with
  balancing.
- Oversampling via `max_ood_size_per_class` draws extra subdivisions from
  the same test-set sources. This only yields *distinct* triangulations for
  randomized subdivisions (`graded`, or `stellar` with `fraction < 1`).
  Barycentric — and stellar with `fraction=1.0` — are deterministic, so an
  oversampled class then contains **exact duplicates** of the same
  subdivided triangulation (also flagged by a `UserWarning`).

In [ ]:
# A balanced benchmark: every class ends up with exactly target_count
# triangulations of at most max_vertices vertices. Balancing needs
# augmentation headroom under the cap: a source triangulation is only
# eligible if n_vertices + n_moves <= max_vertices, and MANTRA's 2D
# minimal triangulations go up to 15 vertices, so a cap of 20 keeps
# every class balanceable with 5 Pachner moves per augmented sample.
balancing = dict(
    balanced=True,
    max_vertices=20,
    balance_kwargs=dict(
        target_count=100,  # samples per class after balancing
        n_moves=5,  # Pachner moves per augmented sample
        use_topology_changes=False,  # don't synthesize missing classes
    ),
    stratified=True,  # keep the splits balanced, too
)

balanced_train = MANTRADivided(
    root="./data", dimension=2, version="latest",
    split_type="train", **balancing,
)
Counter(d.name for d in balanced_train)

In [ ]:
# The other two schemes only change `division_type` and its argument.
# We reuse the balanced configuration from above and cap the OOD split
# at 100 samples per class. For the graded subdivision, vertex_number
# must exceed max_vertices, which guarantees that every OOD sample is
# strictly larger than anything in train/val/test.
for division_type, kwargs in [
    ("barycentric", {"round": 1}),
    ("stellar", {"fraction": 1.0}),
    ("graded", {"vertex_number": 21}),
]:
    test = MANTRADivided(
        root="./data", dimension=2, version="latest", **balancing,
        max_ood_size_per_class=100,
        split_type="test", division_type=division_type, **kwargs,
    )
    ood = MANTRADivided(
        root="./data", dimension=2, version="latest", **balancing,
        max_ood_size_per_class=100,
        split_type="ood", division_type=division_type, **kwargs,
    )
    print(
        f"{division_type:11s}: test[0] {int(test[0].n_vertices):3d} vertices "
        f"-> ood[0] {int(ood[0].n_vertices):3d} vertices"
    )

In [54]:
test = MANTRADivided(
        root="./data", dimension=2, version="latest", **balancing,
        split_type="test", division_type=division_type, **kwargs,
    )
ood = MANTRADivided(
        root="./data", dimension=2, version="latest", **balancing,
        split_type="ood", division_type="barycentric", round=2,
    )
print(ood[0]["n_vertices"])

Processing...
Subdividing OOD (T^2): 100%|██████████| 100/100 [00:00<00:00, 669.83it/s]
Done!


tensor([820])


## 11. Other variants

The base class loads several variants by flipping a keyword argument:

```python
# 3-manifolds instead of surfaces
ManifoldTriangulations(root="./data", dimension=3, version="latest")

# The class-balanced variant, computed on the fly during processing
# (Pachner-move augmentation + deduplication; cached after the first
# run, which can take a while for the 3D dataset). Tune it with
# balance_kwargs, e.g. dict(target_count=500).
ManifoldTriangulations(root="./data", dimension=2, balanced=True)

# Pin a specific released version for reproducibility
ManifoldTriangulations(root="./data", dimension=2, version="v0.0.19")

# Use a local JSON file instead of downloading (e.g. for testing)
ManifoldTriangulations(root="./data", dimension=2, local_path="my_manifolds.json")
```

## Where to go next

- [Visualizing the MANTRA dataset](visualize_data.ipynb)
- [Adding new tasks to MANTRA](adding_new_task.ipynb)
- [Training a GNN on MANTRA](train_gnn.ipynb)